In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
#import os
#os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
#import torch


In [3]:
import os
code_dir = "/content/drive/MyDrive/Colab Notebooks/Occlusion/"
os.chdir(code_dir)

print(os.getcwd())



/content/drive/MyDrive/Colab Notebooks/Occlusion


In [ ]:
local_tmp_dir = "/content/tmp_dataset"
os.makedirs(local_tmp_dir, exist_ok=True)
dest_data_dir = "/content/drive/MyDrive/Colab Notebooks/Occlusion/datasets/"
!unzip -q "/content/drive/MyDrive/Colab Notebooks/Occlusion/datasets/final_dataset.zip" -d "/content/tmp_dataset"

In [ ]:
!ls "/content/tmp_dataset/final_dataset"

 image_metadata.csv  'Padded 384'  'Padded 640'   unpadded


In [ ]:
!cp -r "/content/tmp_dataset/" "/content/drive/MyDrive/Colab Notebooks/Occlusion/datasets/"


In [ ]:
!pip install --upgrade torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time
import os
import matplotlib.pyplot as plt
from torchvision.datasets import ImageFolder
from CustomBackbone import BoTNetBackbone, CustomDataSetModel

# Hyperparameters
in_channels = 128
cbam_reduction = 16
heads = 4
mlp_ratio = 2

learning_rate = 0.001
weight_decay = 0.0001

num_epochs = 30  # Increased epochs for better training
batch_size = 64
train_batch_size = 16
val_batch_size = 8
eval_batch_size = 4

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Paths
model_dir = "/content/drive/MyDrive/Colab Notebooks/Occlusion/model/Pretrained/"
checkpoint_path = os.path.join(model_dir, 'botnet_custom_best.pth')
data_dir = "/content/drive/MyDrive/Colab Notebooks/Occlusion/datasets/final_dataset/"
folder_name = 'unpadded/'

# Paths to training and validation data
train_data_path = data_dir + folder_name + 'train/'
val_data_path = data_dir + folder_name + 'test/'

# Transforms for training and validation
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.5),
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Dataset and DataLoader
train_dataset = ImageFolder(root=train_data_path, transform=transform_train)
val_dataset = ImageFolder(root=val_data_path, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=val_batch_size, shuffle=False, num_workers=2, pin_memory=True)

# Model
botnet_backbone = BoTNetBackbone(in_channels=in_channels, cbam_reduction=cbam_reduction, heads=heads, mlp_ratio=mlp_ratio)
model = CustomDataSetModel(botnet_backbone, num_classes=7)

model.to(device)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Learning rate scheduler
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

# AMP GradScaler
scaler = torch.cuda.amp.GradScaler()

# Checkpoint function
def save_checkpoint(epoch, model, optimizer, loss, checkpoint_path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, checkpoint_path)
    print(f"✅ Saved checkpoint at epoch {epoch + 1} to {checkpoint_path}")

# Training loop with loss/accuracy tracking
def train(model, trainloader, criterion, optimizer, device, num_epochs, checkpoint_path):
    best_acc = 0  # Keep track of best validation accuracy
    train_losses = []  # Track training loss per epoch
    val_accuracies = []  # Track validation accuracy per epoch
    train_accuracies = []  # Track training accuracy per epoch

    # Check if checkpoint exists and resume from it
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])  # Load model state
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])  # Load optimizer state
        start_epoch = checkpoint['epoch'] + 1  # Resume from the next epoch
        print(f"✅ Resuming training from epoch {start_epoch}")
    else:
        start_epoch = 0  # Start from scratch if no checkpoint is found
        print("Starting from scratch...")

    # Training loop starts from the resume epoch
    for epoch in range(start_epoch, num_epochs):
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        model.train()  # Set model to training mode
        for i, data in enumerate(trainloader):
            inputs, labels = data[0].to(device), data[1].to(device)

            optimizer.zero_grad()

            # Forward pass
            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            # Backward pass and optimizer step
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

            if i % 200 == 0:
                print(f'[{epoch + 1}, {i + 1}] loss: {running_loss / 200:.3f}')
                running_loss = 0.0

        # Training accuracy for the epoch
        train_acc = 100 * correct_train / total_train
        train_accuracies.append(train_acc)

        # Learning rate scheduler step
        scheduler.step()

        # Validate at the end of each epoch
        val_acc = evaluate(model, val_loader, device)  # Call to evaluate function
        val_accuracies.append(val_acc)

        # Save the best model based on validation accuracy
        if val_acc > best_acc:
            best_acc = val_acc
            save_checkpoint(epoch, model, optimizer, loss, checkpoint_path)

        # Store the training loss for the epoch
        train_losses.append(running_loss / len(trainloader))

    print(f'Finished Training. Best Accuracy: {best_acc:.2f}%')

    # Plot the loss and accuracy curves
    plot_training_curves(train_losses, train_accuracies, val_accuracies)


# Evaluation loop
def evaluate(model, testloader, device):
    model.eval()  # Set model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            images, labels = data[0].to(device), data[1].to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy of the network on the validation set: {accuracy:.2f}%')
    return accuracy

# Plot loss and accuracy curves
def plot_training_curves(train_losses, train_accuracies, val_accuracies):
    epochs = range(1, len(train_losses) + 1)

    # Loss plot
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label="Training Loss")
    plt.title("Training Loss Over Epochs")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.grid(True)

    # Accuracy plot
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_accuracies, label="Training Accuracy")
    plt.plot(epochs, val_accuracies, label="Validation Accuracy")
    plt.title("Training and Validation Accuracy Over Epochs")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy (%)")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# Start training and evaluation
train(model, train_loader, criterion, optimizer, device, num_epochs, checkpoint_path)


<ipython-input-4-d84c1bc7fec6>:75: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Starting from scratch...


<ipython-input-4-d84c1bc7fec6>:117: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[1, 1] loss: 0.010


In [ ]:
import os
os.sync()


In [ ]:
!ls -lh "/content/drive/MyDrive/Colab Notebooks/Occlusion/model/Pretrained/"
!mkdir -p "/content/drive/MyDrive/Colab Notebooks/Occlusion/model/Pretrained/new"
!cp "/content/drive/MyDrive/Colab Notebooks/Occlusion/model/Pretrained/botnet_custom.pth" "/content/drive/MyDrive/Colab Notebooks/Occlusion/model/Pretrained/new"

total 70M
-rw------- 1 root root 5.2M Apr 27 17:15 botnet_cifar10.pth
-rw------- 1 root root 4.4M Apr 27 21:58 botnet_custom.pth
drwx------ 2 root root 4.0K Apr 27 20:01 classification_old
-rw------- 1 root root  31M Apr 26 21:25 full_occlusion_custom_backbone_botnet_model.pth
-rw------- 1 root root  31M Apr 26 21:25 full_occlusion_custom_backbone_botnet_model_weights.pth


In [ ]:
!ls -lh "/content/drive/MyDrive/Colab Notebooks/Occlusion/model/Pretrained/new"

ls: cannot access '/content/drive/MyDrive/Colab Notebooks/Occlusion/model/Pretrained/new': No such file or directory


In [ ]:
import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


In [ ]:
import os
os.chdir("/content/drive/MyDrive/Colab Notebooks/Occlusion/model/Pretrained/")
print(os.getcwd())
!ls botnet_custom*


/content/drive/MyDrive/Colab Notebooks/Occlusion/model/Pretrained
botnet_custom.pth
